# 05 — Statistics & Hypotheses

Implements Stage 5 (`09_ETL_AND_ANALYSIS_ENGINE.md`) exactly as `04_STATISTICAL_ANALYSIS_PLAN.md` specifies: simple linear trend (year -> metric) and Pearson correlation (Spearman as a secondary check), significance at **alpha = 0.05**. No multiple regression, no multiple-comparison correction, no residual diagnostics.

> All calculations are imported from `src/`; this notebook only orchestrates and reports (`11_CODE_STRUCTURE.md`).

**All data is real: eBird EBD v1.16 (IN-GJ, May 2026) for birds, and ERA5/HOURLY + MODIS via Google Earth Engine for the environment. H1 (temperature) now uses real winter temperatures.**

In [1]:
# --- Setup: make src/ importable (works whether cwd is repo root or notebooks/) ---
import os, sys, datetime, platform
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "requirements.txt")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
REPO_ROOT = _root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np, pandas as pd, scipy
import matplotlib
matplotlib.use("Agg")            # headless: write figure files, no GUI needed
import matplotlib.pyplot as plt

from src import load_and_clean, observer_effort, migration_metrics
from src import environmental_data as envmod
from src import statistics as stats_
from src import validation as V

FIG_DIR = os.path.join(REPO_ROOT, "outputs", "figures")
TAB_DIR = os.path.join(REPO_ROOT, "outputs", "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)

# All inputs are real: eBird EBD v1.16 for birds, ERA5/HOURLY + MODIS via Google
# Earth Engine for the environment (H1 temperature is real).
DATA_NOTE = "Real: eBird EBD v1.16 (IN-GJ, May 2026) + ERA5/MODIS via GEE"
STUDY_PERIOD = "2010-2025"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("setup complete; REPO_ROOT =", REPO_ROOT)

setup complete; REPO_ROOT = /home/tops/Documents/BirdSense


In [2]:
# --- Reproducibility header (04_STATISTICAL_ANALYSIS_PLAN.md) ---
print("Dataset version :", "eBird EBD v1.16 (IN-GJ, relMay-2026); ENV=ERA5/HOURLY + MODIS (GEE)")
print("Analysis date   :", datetime.date.today().isoformat())
print("Python          :", platform.python_version())
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

Dataset version : eBird EBD v1.16 (IN-GJ, relMay-2026); ENV=ERA5/HOURLY + MODIS (GEE)
Analysis date   : 2026-07-14
Python          : 3.10.12
pandas 2.3.3 | numpy 2.2.6 | scipy 1.15.3 | matplotlib 3.10.9


In [3]:
# --- Run Stages 1-4 from src (no metric logic here; all imported) ---
stage1     = load_and_clean.run_stage1()
effort     = observer_effort.compute_observer_effort(stage1.clean_observations, stage1.clean_checklists)
metrics    = migration_metrics.compute_migration_metrics(stage1.clean_observations, stage1.clean_checklists)
# Real annual winter environment (ERA5/HOURLY + MODIS via GEE) is cached; load it.
_env_csv = os.path.join(REPO_ROOT, "data", "processed", "environmental_annual.csv")
if os.path.exists(_env_csv):
    annual_env = pd.read_csv(_env_csv)                       # REAL (GEE, cached)
    ENV_SOURCE = "REAL (ERA5/MODIS via GEE, cached)"
else:
    annual_env = envmod.build_annual_environmental(mock=True)  # fallback: FAKE
    ENV_SOURCE = "MOCK (no cached GEE table)"
print("metrics", metrics.shape, "| effort", effort.shape,
      "| annual_env", annual_env.shape, "| env:", ENV_SOURCE)

metrics (192, 14) | effort (192, 6) | annual_env (16, 5) | env: REAL (ERA5/MODIS via GEE, cached)


## Trend analysis (H2): confirmed arrival & departure vs year
Per-species linear regression; low-confidence species-years excluded (`02_METRICS_METHODOLOGY.md` sec 1).

In [4]:
arrival_trend = stats_.species_metric_trend(metrics, "first_arrival")
arrival_trend

,species,common,habitat,slope,intercept,r_squared,p_value,n
0,Anas acuta,Northern Pintail,wetland,-0.523529,1059.345588,0.424065,0.006286,16
1,Spatula clypeata,Northern Shoveler,wetland,-0.366176,741.073529,0.416575,0.006929,16
2,Spatula querquedula,Garganey,wetland,-0.973529,1969.345588,0.333926,0.019056,16
3,Mareca penelope,Eurasian Wigeon,wetland,-0.714706,1445.669118,0.600946,0.000419,16
4,Aythya ferina,Common Pochard,wetland,-1.194118,2415.257353,0.430657,0.005764,16
5,Anser indicus,Bar-headed Goose,grassland_dryland,-1.644118,3325.132353,0.570226,0.000720,16
6,Anser anser,Greylag Goose,wetland,-0.886765,1793.360294,0.464620,0.003638,16
7,Grus grus,Common Crane,grassland_dryland,-0.411765,833.360294,0.390166,0.009688,16
8,Grus virgo,Demoiselle Crane,grassland_dryland,-0.867647,1755.227941,0.575182,0.000661,16
9,Phoenicopterus roseus,Greater Flamingo,wetland,-0.544118,1101.007353,0.371446,0.012199,16


In [5]:
departure_trend = stats_.species_metric_trend(metrics, "last_departure")
departure_trend

,species,common,habitat,slope,intercept,r_squared,p_value,n
0,Anas acuta,Northern Pintail,wetland,0.927941,-1509.808824,0.322629,0.021713,16
1,Spatula clypeata,Northern Shoveler,wetland,0.919118,-1491.882353,0.295517,0.029516,16
2,Spatula querquedula,Garganey,wetland,1.041176,-1739.198529,0.357928,0.014359,16
3,Mareca penelope,Eurasian Wigeon,wetland,1.039706,-1735.669118,0.385687,0.010243,16
4,Aythya ferina,Common Pochard,wetland,0.979412,-1613.963235,0.355277,0.014821,16
5,Anser indicus,Bar-headed Goose,grassland_dryland,14.416176,-28763.698529,0.341914,0.017358,16
6,Anser anser,Greylag Goose,wetland,1.050000,-1756.500000,0.394683,0.009156,16
7,Grus grus,Common Crane,grassland_dryland,0.811765,-1274.985294,0.303176,0.027087,16
8,Grus virgo,Demoiselle Crane,grassland_dryland,1.294118,-2250.257353,0.546592,0.001065,16
9,Phoenicopterus roseus,Greater Flamingo,wetland,0.952941,-1560.808824,0.354481,0.014962,16


In [6]:
n_sig_arr = int(arrival_trend["p_value"].apply(stats_.is_significant).sum())
n_sig_dep = int(departure_trend["p_value"].apply(stats_.is_significant).sum())
print(f"Arrival trends significant at alpha={stats_.ALPHA}: {n_sig_arr}/{len(arrival_trend)}")
print(f"Departure trends significant at alpha={stats_.ALPHA}: {n_sig_dep}/{len(departure_trend)}")

Arrival trends significant at alpha=0.05: 12/12
Departure trends significant at alpha=0.05: 12/12


## Correlation analysis (H1): winter temperature vs arrival
Pearson r + p per species (Spearman shown as the secondary check).

In [7]:
corr = stats_.build_correlation_table(metrics, annual_env)
corr

,common_name,scientific_name,n_years,pearson_r,p_value,spearman_rho,interpretation
0,Northern Pintail,Anas acuta,16,-0.424,0.1018,-0.254,none
1,Northern Shoveler,Spatula clypeata,16,-0.282,0.2894,-0.202,none
2,Garganey,Spatula querquedula,16,-0.514,0.0419,-0.310,strong
3,Eurasian Wigeon,Mareca penelope,16,-0.344,0.1925,-0.261,none
4,Common Pochard,Aythya ferina,16,-0.038,0.8889,-0.167,none
5,Bar-headed Goose,Anser indicus,16,-0.317,0.2315,-0.134,none
6,Greylag Goose,Anser anser,16,-0.276,0.3015,-0.195,none
7,Common Crane,Grus grus,16,-0.246,0.3590,-0.240,none
8,Demoiselle Crane,Grus virgo,16,-0.206,0.4445,-0.222,none
9,Greater Flamingo,Phoenicopterus roseus,16,-0.124,0.6465,0.002,none


## Habitat-category comparison (H3)
Category-level arrival trend: wetland-dependent vs grassland/dryland (comparative/descriptive, no formal group-difference test at this scope).

In [8]:
hab_year, hab_trends = stats_.habitat_category_trends(metrics)
for hab, tr in hab_trends.items():
    print(hab, "-> slope", None if tr["slope"] is None else round(tr["slope"],3),
          "days/yr, p =", None if tr["p_value"] is None else round(tr["p_value"],3))

grassland_dryland -> slope -0.975 days/yr, p = 0.0
wetland -> slope -0.698 days/yr, p = 0.0


## Hypothesis evaluation (H1-H3)
Decisions use alpha = 0.05 and are reported honestly per `04_STATISTICAL_ANALYSIS_PLAN.md`. Both H1 (real ERA5 winter temperature) and H2 (real eBird timing) now run on real data.)

In [9]:
# Interpretation/assembly only -- all statistics come from src.
n_h1 = int((corr["interpretation"] != "none").sum())
h1 = "Supported" if n_h1 > len(corr)/2 else ("Inconclusive" if n_h1 else "Not Supported")
n_h2 = max(n_sig_arr, n_sig_dep)
h2 = "Supported" if n_h2 > len(arrival_trend)/2 else ("Inconclusive" if n_h2 else "Not Supported")
sw = hab_trends.get("wetland", {}).get("slope")
sd = hab_trends.get("grassland_dryland", {}).get("slope")
if sw is None or sd is None:
    h3 = "Inconclusive"
else:
    h3 = "Supported (categories differ)" if (np.sign(sw) != np.sign(sd) or abs(sw-sd) >= 1.0) else "Not Supported (similar)"
print(f"H1 (temperature -> earlier arrival): {h1}  ({n_h1}/{len(corr)} species significant)")
print(f"H2 (timing changed over period)   : {h2}  (max {n_h2}/{len(arrival_trend)} species significant)")
print(f"H3 (habitat categories differ)    : {h3}  (wetland {sw}, dryland {sd} days/yr)")

H1 (temperature -> earlier arrival): Inconclusive  (1/12 species significant)
H2 (timing changed over period)   : Supported  (max 12/12 species significant)
H3 (habitat categories differ)    : Not Supported (similar)  (wetland -0.6978758169934642, dryland -0.9745098039215686 days/yr)


## Effort-robustness comparison — is H1/H2 real or observer artifact?
Confirmed arrival is **effort-sensitive** (calendar-year 2nd observation collapses toward Jan 1 as the ~30x observer growth fills early January). **Peak week is detection-rate-weighted**, so effort-robust. If the signal survives with peak week, it is more likely real phenology. Arrival results above are kept as-is; this only adds the robust view. All metrics from `src` (04: correlation + linear trend only, no multiple regression).

### (1) Effort confound diagnostic: annual effort vs confirmed arrival

In [10]:
effort_vs_arrival = stats_.effort_vs_arrival_correlation(metrics, effort)
effort_vs_arrival

,common_name,scientific_name,n_years,effort_vs_arrival_r,p_value,interpretation
0,Northern Pintail,Anas acuta,16,-0.447,0.0825,none
1,Northern Shoveler,Spatula clypeata,16,-0.396,0.1293,none
2,Garganey,Spatula querquedula,16,-0.420,0.1050,none
3,Eurasian Wigeon,Mareca penelope,16,-0.499,0.0489,weak
4,Common Pochard,Aythya ferina,16,-0.471,0.0655,none
5,Bar-headed Goose,Anser indicus,16,-0.545,0.0290,strong
6,Greylag Goose,Anser anser,16,-0.432,0.0944,none
7,Common Crane,Grus grus,16,-0.413,0.1118,none
8,Demoiselle Crane,Grus virgo,16,-0.525,0.0368,strong
9,Greater Flamingo,Phoenicopterus roseus,16,-0.399,0.1259,none


### (2) H1 side by side: temperature vs arrival AND vs peak week

In [11]:
h1_compare = stats_.build_h1_comparison(metrics, annual_env)
h1_compare

,common_name,scientific_name,temp_vs_arrival_r,arrival_p,arrival_evidence,temp_vs_peakweek_r,peakweek_p,peakweek_evidence
0,Northern Pintail,Anas acuta,-0.424,0.1018,none,-0.350,0.1845,none
1,Northern Shoveler,Spatula clypeata,-0.282,0.2894,none,0.406,0.1183,none
2,Garganey,Spatula querquedula,-0.514,0.0419,strong,-0.217,0.4199,none
3,Eurasian Wigeon,Mareca penelope,-0.344,0.1925,none,-0.300,0.2593,none
4,Common Pochard,Aythya ferina,-0.038,0.8889,none,-0.216,0.4224,none
5,Bar-headed Goose,Anser indicus,-0.317,0.2315,none,-0.279,0.2961,none
6,Greylag Goose,Anser anser,-0.276,0.3015,none,0.303,0.2545,none
7,Common Crane,Grus grus,-0.246,0.3590,none,-0.117,0.6666,none
8,Demoiselle Crane,Grus virgo,-0.206,0.4445,none,-0.146,0.5896,none
9,Greater Flamingo,Phoenicopterus roseus,-0.124,0.6465,none,-0.177,0.5131,none


### (3) H2 side by side: arrival trend vs peak-week trend

In [12]:
h2_compare = stats_.build_h2_comparison(metrics)
h2_compare

,common_name,scientific_name,arrival_slope_days_per_yr,arrival_p,peakweek_slope_weeks_per_yr,peakweek_p
0,Northern Pintail,Anas acuta,-0.524,0.0063,-2.162,0.0574
1,Northern Shoveler,Spatula clypeata,-0.366,0.0069,-0.572,0.4740
2,Garganey,Spatula querquedula,-0.974,0.0191,-1.219,0.1918
3,Eurasian Wigeon,Mareca penelope,-0.715,0.0004,-0.762,0.4712
4,Common Pochard,Aythya ferina,-1.194,0.0058,-2.010,0.1099
5,Bar-headed Goose,Anser indicus,-1.644,0.0007,-0.081,0.9490
6,Greylag Goose,Anser anser,-0.887,0.0036,0.881,0.5143
7,Common Crane,Grus grus,-0.412,0.0097,-1.824,0.1260
8,Demoiselle Crane,Grus virgo,-0.868,0.0007,-1.685,0.0996
9,Greater Flamingo,Phoenicopterus roseus,-0.544,0.0122,-1.684,0.0704


In [13]:
# Effort-robust summary counts (interpretation only; stats from src)
n_arr_sig = int((h1_compare["arrival_p"] < stats_.ALPHA).sum())
n_pk_sig  = int((h1_compare["peakweek_p"] < stats_.ALPHA).sum())
n_confound = int((effort_vs_arrival["p_value"] < stats_.ALPHA).sum())
print(f"H1 temp->arrival significant : {n_arr_sig}/12")
print(f"H1 temp->peak-week significant: {n_pk_sig}/12  (effort-robust metric)")
print(f"effort->arrival significant  : {n_confound}/12  (confound present where high)")

H1 temp->arrival significant : 1/12
H1 temp->peak-week significant: 0/12  (effort-robust metric)
effort->arrival significant  : 3/12  (confound present where high)


_Read honestly: where arrival is significant but **peak week is not** and **effort-vs-arrival is strong**, the arrival result is likely an observer-effort artifact, not phenology. Where peak week also holds, the signal is more credible._

_Conclusions above are computed live from real data (eBird observations + ERA/MODIS via GEE). A 'no meaningful association' outcome is a valid, reportable finding (`01_RESEARCH_METHODOLOGY.md`)._